In [ ]:
import scvi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

In [ ]:
import torch
print(torch.__file__)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import bbknn

import celltypist
from celltypist import models
from celltypist.models import Model

# multiomics data import

In [ ]:
adata_scrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_01.h5ad")
adata_scraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad")

In [ ]:
adata_snraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_raw.h5ad")
adata_snrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_adata_01.h5ad")

In [ ]:
adata_spraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/Cosmx_v1/spatial_raw.h5ad")
adata_sprna=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/Cosmx_v1/cosmx_adata_01.h5ad")

In [ ]:
adata_atlas=sc.read('/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/public/human_liver_atlas.h5ad')

In [ ]:
adata_scrna

In [ ]:
adata_snrna

In [ ]:
adata_sprna

In [ ]:
adata_atlas

In [ ]:
adata_atlas.obs.to_csv("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/atlas_obs.csv")

In [ ]:
adata_atlas.obs

In [ ]:
adata_atlas_sc = adata_atlas[adata_atlas.obs['typeSample'].isin(['scRnaSeq'])].copy()

In [ ]:
adata_scrna.obs

In [ ]:
adata_sc = sc.concat(
    [adata_scraw, adata_atlas_sc],
    join='inner',       # ✅ 공통 유전자만
    label='batch',      # batch 구분 컬럼
    keys=['pre', 'atlas'],
    index_unique=None
)
print(adata_sc.shape)

In [ ]:
adata_sc.raw = adata_sc

In [ ]:
# 3개 방식 합치기 
import scanpy as sc

adata_merged = sc.concat(
    [adata_scraw, adata_snraw, adata_spraw],
    join='inner',       # ✅ 공통 유전자만
    label='batch',      # batch 구분 컬럼
    keys=['scRNA', 'snRNA', 'CosMx'],
    index_unique=None
)
print(adata_merged.shape)

In [ ]:
# 3개 방식 합치기 
import scanpy as sc

adata_merged = sc.concat(
    [adata_scraw, adata_snraw],
    join='inner',       # ✅ 공통 유전자만
    label='batch',      # batch 구분 컬럼
    keys=['scRNA', 'snRNA'],
    index_unique=None
)
print(adata_merged.shape)

In [ ]:
adata_merged.raw = adata_merged

In [ ]:
adata_merged.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_01.h5ad")

In [ ]:
adata_merged.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/scsn_merged_adata_01.h5ad")

# sc/sn merged analysis

In [ ]:
adata_merged=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/scsn_merged_adata_01.h5ad")

In [ ]:
adata_merged

In [ ]:
sc.pp.normalize_total(adata_merged, target_sum=1e4)
sc.pp.log1p(adata_merged)

# 변수 유전자 설정 
sc.pp.highly_variable_genes(adata_merged, flavor='seurat', n_top_genes=2000)

In [ ]:
sc.pp.scale(adata_merged, max_value=10)
# Run PCA
sc.tl.pca(adata_merged, svd_solver='arpack')
print(adata_merged.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_merged, color='sample')

import matplotlib.pyplot as plt
var_ratio = adata_merged.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata_merged, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_merged, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_merged)
sc.pl.umap(adata_merged, color=['sample'])

In [ ]:
adata_merged

In [ ]:
sc.pl.umap(adata_merged, color=['batch'])

In [ ]:
adata_merged.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/scsn_merged_adata_02.h5ad")

In [ ]:
adata_merged=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/scsn_merged_adata_02.h5ad")
adata_merged

In [ ]:
bbknn.bbknn(adata_merged, batch_key='batch')
sc.tl.umap(adata_merged)
sc.pl.umap(adata_merged, color=['batch'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_merged.obsm['X_pca'],adata_merged.obs, 'batch')

# Harmony 결과를 저장
adata_merged.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_merged, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_merged)
sc.pl.umap(adata_merged, color=['batch'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_merged.obsm['X_pca'],adata_merged.obs, 'sample')

# Harmony 결과를 저장
adata_merged.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_merged, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_merged)
sc.pl.umap(adata_merged, color=['batch'])

In [ ]:
import torch
import scvi
print(torch.cuda.is_available())

In [ ]:
print("cuda available:", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())
print("device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA device")
print("torch version:", torch.__version__)
print("cuda version:", torch.version.cuda)

In [ ]:
adata_merged_scvi=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/scsn_merged_adata_01.h5ad")
adata_merged_scvi

In [ ]:
adata_merged_scvi.obs_names_make_unique

In [ ]:
sc.pp.highly_variable_genes(adata_merged_scvi, flavor='cell_ranger', n_top_genes=2000)

In [ ]:
# scVI 모델 학습
scvi.model.SCVI.setup_anndata(adata_merged_scvi, batch_key="batch")
model = scvi.model.SCVI(adata_merged_scvi)
model.train()

In [ ]:
import scvi
print(scvi.__version__) 

In [ ]:
adata_merged_scvi.obsm["X_scVI"] = model.get_latent_representation()

In [ ]:
adata_merged_scvi.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/scsn_merged_adata_scvi01.h5ad")

In [ ]:
adata_merged_scvi

In [ ]:
sc.pp.neighbors(adata_merged_scvi, use_rep="X_scVI")
sc.tl.umap(adata_merged_scvi)

In [ ]:
sc.pl.umap(adata_merged_scvi, color = 'state')

In [ ]:
sc.pl.umap(adata_merged_scvi, color = 'batch')

# sc/sn/sp merged analysis

In [ ]:
adata_merged=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_01.h5ad")

In [ ]:
sc.pp.highly_variable_genes(
    adata_merged,
    n_top_genes=2000,         # 원하는 수
    flavor='seurat_v3',       # sc/snRNA 혼합 시 seurat_v3 추천
    batch_key='batch'         # ✅ batch별로 HVG 추출 후 통합
)

In [ ]:
adata_merged

In [ ]:
#adata_merged = adata_merged[:, adata_merged.var['highly_variable']]

In [ ]:
adata_merged.var['highly_variable']

In [ ]:
# scRNA hvg 만 이용하는경우
sc.pp.highly_variable_genes(adata_scraw, n_top_genes=2000, flavor='seurat_v3')
hvg_list = adata_scraw.var[adata_scraw.var['highly_variable']].index

In [ ]:
merged_hvg = set(adata_merged.var_names[adata_merged.var['highly_variable']])
scraw_hvg = set(adata_scraw.var_names[adata_scraw.var['highly_variable']])

common_hvg = merged_hvg & scraw_hvg
only_merged = merged_hvg - scraw_hvg
only_scraw = scraw_hvg - merged_hvg

print(f"공통 HVG 수: {len(common_hvg)}")
print(f"병합 후 HVG 전용 수: {len(only_merged)}")
print(f"scRNA HVG 전용 수: {len(only_scraw)}")

In [ ]:
# 1 merged hvg 이용한 분석

In [ ]:
sc.pp.normalize_total(adata_merged, target_sum=1e4)
sc.pp.log1p(adata_merged)

In [ ]:
adata_merged.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_02.h5ad")

In [ ]:
adata_merged=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_02.h5ad")

In [ ]:
adata_merged

In [ ]:
sc.pp.scale(adata_merged, max_value=10)
sc.tl.pca(adata_merged, svd_solver='arpack')
print(adata_merged.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_merged, color='batch')

import matplotlib.pyplot as plt
var_ratio = adata_merged.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata_merged, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_merged, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_merged)
sc.pl.umap(adata_merged, color=['batch'])

In [ ]:
adata_merged.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_03.h5ad")

In [ ]:
adata_bbknn = adata_merged

In [ ]:
adata_harmony = adata_merged

In [ ]:
import bbknn
bbknn.bbknn(adata_bbknn, batch_key='batch')
sc.tl.umap(adata_bbknn)

# Figure 생성
fig, ax = plt.subplots(figsize=(5, 5))

sc.pl.umap(
    adata_bbknn, 
    color='batch',
    size=20,
    alpha=0.8,
    legend_fontsize=12,
    ax=ax  
)

plt.show()

In [ ]:
adata_bbknn.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_bbknn.h5ad")

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_harmony.obsm['X_pca'],adata_harmony.obs, 'batch')

# Harmony 결과를 저장
adata_harmony.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_harmony, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_harmony)
sc.pl.umap(adata_harmony, color=['batch'])

In [ ]:
adata_harmony.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_harmony.h5ad")

In [ ]:
adata_harmony=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_harmony.h5ad")

In [ ]:
adata_harmony

In [ ]:
adata_merged=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_02.h5ad")

In [ ]:
adata_merged

In [ ]:
celltypist.models.models_description()

In [ ]:
adata_merged.obs_names_make_unique()

In [ ]:
predictions = celltypist.annotate(adata_merged, model = 'Healthy_Human_Liver.pkl', majority_voting = True)

In [ ]:
adata_merged = predictions.to_adata()

In [ ]:
adata_merged.obs

In [ ]:
adata_harmony.obs['celltypist_liver'] = adata_merged.obs['majority_voting']

In [ ]:
sc.pl.umap(adata_harmony, color=['celltypist_liver'])

In [ ]:
adata_harmony.obs['celltypist_liver'].value_counts()

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 차원 축소된 데이터 (예: PCA or UMAP 등)
X = adata_harmony.obsm['X_pca']  # 또는 'X_umap' 등

inertias = []
k_range = range(2, 30)  

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)  # inertia = 클러스터 내부 거리 제곱합

# Plot
plt.figure(figsize=(6, 4))
plt.plot(k_range, inertias, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.grid(True)
plt.show()

In [ ]:
sc.tl.leiden(adata_harmony, resolution=0.5, key_added="10cluster")
sc.pl.umap(adata_harmony, color=['10cluster'])

In [ ]:
adata_hm=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_harmony.h5ad")

In [ ]:
adata_hm=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_harmony.h5ad")
adata_hm

In [ ]:
adata_hm=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_harmony.h5ad")
sc.pl.umap(adata_hm, color=['batch'])

In [ ]:
sc.pl.umap(adata_hm, color=['batch'])

In [ ]:
adata_hm

In [ ]:
adata_hm

In [ ]:
sc.pl.umap(adata_hm, color=['state'])

In [ ]:
adata_hm.obs['state_specific'] = adata_hm.obs['state']
adata_hm.obs['state'] = adata_hm.obs['state'].replace({
    'Normal': 'Healthy',
    'Fibrosis': 'MASLD',
    'Inflammation1': 'MASLD',
    'Inflammation2': 'MASLD',
    'NAFLD': 'MASLD',
    'Normal_DM': 'Healthy',
    'Steatosis': 'MASLD',
    'healthy': 'Healthy',
    'normal': 'Healthy'
})
sc.pl.umap(adata_hm, color=['state'])

In [ ]:
adata_hm=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_harmony01.h5ad")

In [ ]:
adata_hm=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_harmony01.h5ad")
adata_hm

In [ ]:
adata_hm.obs_names_make_unique

In [ ]:
sc.pl.umap(adata_hm, color=['batch'])

In [ ]:
adata_hm.obs['celltypist_liver'] = adata_merged.obs['majority_voting']

In [ ]:
sc.pl.umap(adata_hm, color=['celltypist_liver'])

In [ ]:
sc.pl.umap(
    adata_hm,
    color='celltypist_liver',
    frameon=False,
    title='celltypist_liver',
    size=6,
    show=False,        # 이걸 False로 해야 plt.savefig()가 동작
    #return_fig=True    # <- 중요: 그림 객체 반환
)
# 직접 저장
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_IL25.pdf", bbox_inches='tight', dpi=300)
#plt.close()

In [ ]:
sc.pl.umap(
    adata_hm,
    color='batch',
    frameon=False,
    title='analysis',
    size=6,
    show=False,        # 이걸 False로 해야 plt.savefig()가 동작
    #return_fig=True    # <- 중요: 그림 객체 반환
)
# 직접 저장
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_IL25.pdf", bbox_inches='tight', dpi=300)
#plt.close()

In [ ]:
sc.tl.leiden(adata_hm, resolution=0.5, key_added="cluster")
sc.pl.umap(adata_hm, color=['cluster'])

In [ ]:
sc.pl.umap(adata_hm, color=['cluster'],legend_loc='on data')

In [ ]:
adata_hm.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_harmony01.h5ad")

In [ ]:
adata_merged=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_02.h5ad")

In [ ]:
adata_hm.raw=adata_merged

In [ ]:
sc.tl.dendrogram(adata_hm,groupby='cluster')
sc.tl.rank_genes_groups(adata_hm, groupby='cluster',use_raw=True, method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_hm, n_genes=20)

In [ ]:
rank_df = sc.get.rank_genes_groups_df(adata_hm, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_hm,
    var_names=top_genes,
    groupby="cluster",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
rank_df

In [ ]:
rank_df.to_csv("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/27cluster_fam_markers_250730.csv")

In [ ]:
adata_hm.obs

In [ ]:
sc.tl.rank_genes_groups(
    adata_hm,
    groupby="state",        # 비교할 obs 열
    groups=["MASLD"],     # 비교할 그룹
    reference="Healthy",     # 참조 그룹
    use_raw=True,
    method="wilcoxon"       # 또는 't-test', 'logreg'
)

In [ ]:
deg_df = sc.get.rank_genes_groups_df(adata_hm, group="MASLD")
print(deg_df.head())

In [ ]:
deg_df.to_csv("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/27cluster_MASLD_markers_250730.csv")

In [ ]:
# -log10(padj) 계산
deg_df['-log10(padj)'] = -np.log10(deg_df['pvals_adj'])

# inf 값 또는 너무 큰 값 제한 (예: 300)
max_logp = 300
deg_df['-log10(padj)'] = np.minimum(deg_df['-log10(padj)'], max_logp)
genes_of_interest = ["SLC7A5"]

plt.figure(figsize=(7, 6))
plt.scatter(
    deg_df['logfoldchanges'], 
    deg_df['-log10(padj)'], 
    c="grey", s=10, alpha=0.7
)

# 유의미한 cutoff 설정 (예: logFC > 1, padj < 0.05)
sig_up = (deg_df['logfoldchanges'] > 0.58) & (deg_df['pvals_adj'] < 0.05)
sig_down = (deg_df['logfoldchanges'] < -0.58) & (deg_df['pvals_adj'] < 0.05)

plt.scatter(deg_df.loc[sig_up, 'logfoldchanges'],
            deg_df.loc[sig_up, '-log10(padj)'],
            c="red", s=10, label="Upregulated")

plt.scatter(deg_df.loc[sig_down, 'logfoldchanges'],
            deg_df.loc[sig_down, '-log10(padj)'],
            c="blue", s=10, label="Downregulated")

# cutoff 선
plt.axvline(0.58, color="black", linestyle="--")
plt.axvline(-0.58, color="black", linestyle="--")
plt.axhline(-np.log10(0.05), color="black", linestyle="--")

plt.xlabel("log2 Fold Change")
plt.ylabel("-log10 Adjusted p-value")
plt.title("Volcano Plot: Disease vs Normal")
plt.legend()

for gene in genes_of_interest:
    if gene in deg_df['names'].values:
        x = deg_df.loc[deg_df['names'] == gene, 'logfoldchanges'].values[0]
        y = deg_df.loc[deg_df['names'] == gene, '-log10(padj)'].values[0]
        plt.text(x, y, gene, fontsize=8, color="black")  # 텍스트 표시


plt.tight_layout()
plt.show()

# scRNA merge analysis

In [ ]:
adata_sc.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/scRNA_01.h5ad")

In [ ]:
sc.pp.normalize_total(adata_sc, target_sum=1e4)
sc.pp.log1p(adata_sc)

In [ ]:
adata_sc.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/scRNA_02.h5ad")

In [ ]:
adata_sc=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/scRNA_02.h5ad")

In [ ]:
adata_sc

In [ ]:
# 변수 유전자 설정 
sc.pp.highly_variable_genes(adata_sc, flavor='seurat', n_top_genes=2000)

# Scale data
sc.pp.scale(adata_sc, max_value=10)
# Run PCA
sc.tl.pca(adata_sc, svd_solver='arpack')
print(adata_sc.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_sc, color='batch')

import matplotlib.pyplot as plt
var_ratio = adata_sc.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata_sc, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_sc, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_sc)
sc.pl.umap(adata_sc, color=['sample'])

In [ ]:
adata_sc

In [ ]:
adata_sc.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/scRNA_03.h5ad")

In [ ]:
adata_sc=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/scRNA_03.h5ad")
adata_sc

In [ ]:
adata_scrna.obs

In [ ]:
adata_atlas_sc.obs['state'] = "healthy"

In [ ]:
adata_atlas_sc.obs

In [ ]:
adata_sc.obs['state'] = adata_scrna.obs['state']

In [ ]:
adata_sc.obs['state'] = adata_sc.obs['state'].fillna('healthy')

In [ ]:
adata_sc.obs

In [ ]:
sc.pl.umap(adata_sc, color = 'state')

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_sc.obsm['X_pca'],adata_sc.obs, 'sample')

# Harmony 결과를 저장
adata_sc.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_sc, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_sc)

In [ ]:
sc.pl.umap(adata_sc, color = 'state')

In [ ]:
adata_sc_raw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/scRNA_02.h5ad")

In [ ]:
# Healthy_Human_Liver.pkl
predictions = celltypist.annotate(adata_sc_raw, model = 'Healthy_Human_Liver.pkl', majority_voting = True)

adata_sc_raw = predictions.to_adata()
adata_sc_raw.obs['celltypist_Immune']= adata_sc_raw.obs['majority_voting']

In [ ]:
adata_sc.obs['celltypist_Liver']= adata_sc_raw.obs['majority_voting']

In [ ]:
sc.pl.umap(adata_sc, color = 'celltypist_Liver')

In [ ]:
adata_sc.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/scRNA_04.h5ad")

# multiomics _ monocyte mp analysis

In [ ]:
adata_hm=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_harmony01.h5ad")

In [ ]:
adata_hm

In [ ]:
adata_mp = adata_hm[adata_hm.obs['cluster'].isin(['9'])].copy()
adata_mp

In [ ]:
adata_merged=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_01.h5ad")
adata_merged.obs

In [ ]:
adata_merged.obs['state']= adata_hm.obs['state']
adata_merged.obs['cluster']= adata_hm.obs['cluster']

In [ ]:
adata_hm.obs

In [ ]:
adata_mp = adata_merged[adata_merged.obs['cluster'].isin(['9'])].copy()
adata_mp

In [ ]:
adata_mp.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_mp01.h5ad")

In [ ]:
sc.pp.normalize_total(adata_mp, target_sum=1e4)
sc.pp.log1p(adata_mp)

In [ ]:
adata_mp.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_mp02.h5ad")

In [ ]:
adata_mp=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_mp02.h5ad")

In [ ]:
sc.pp.highly_variable_genes(adata_mp, flavor='seurat', n_top_genes=2000)
# Scale data
sc.pp.scale(adata_mp, max_value=10)
# Run PCA
sc.tl.pca(adata_mp, svd_solver='arpack')
print(adata_mp.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_mp, color='state')

import matplotlib.pyplot as plt
var_ratio = adata_mp.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata_mp, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_mp, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_mp)
sc.pl.umap(adata_mp, color=['state'])

In [ ]:
sc.pl.umap(adata_mp, color=['batch'])

In [ ]:
import bbknn
adata_mp_bbknn = adata_mp
bbknn.bbknn(adata_mp_bbknn, batch_key='batch')
sc.tl.umap(adata_mp_bbknn)

# Figure 생성
fig, ax = plt.subplots(figsize=(5, 5))

sc.pl.umap(
    adata_mp_bbknn, 
    color='batch',
    size=20,
    alpha=0.8,
    legend_fontsize=12,
    ax=ax  
)

plt.show()

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 차원 축소된 데이터 (예: PCA or UMAP 등)
X = adata_mp_bbknn.obsm['X_pca']  # 또는 'X_umap' 등

inertias = []
k_range = range(2, 30)  

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)  # inertia = 클러스터 내부 거리 제곱합

# Plot
plt.figure(figsize=(6, 4))
plt.plot(k_range, inertias, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.grid(True)
plt.show()


In [ ]:
sc.tl.leiden(adata_mp_bbknn, resolution=0.5, key_added="cluster_mp")
sc.pl.umap(adata_mp_bbknn, color=['cluster_mp'])

In [ ]:
sc.pl.umap(adata_mp_bbknn, color=['cluster_mp','batch','state'])

In [ ]:
sc.pl.tsne(adata_mp_bbknn, color=['cluster_mp','batch'])

In [ ]:
sc.pl.umap(adata_mp_bbknn, color=['VSIG4','CD5L'])

In [ ]:
adata_mp_bbknn.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_mp_bbknn01.h5ad")

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_mp.obsm['X_pca'],adata_mp.obs, 'batch')

# Harmony 결과를 저장
adata_mp.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_mp, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_mp)
sc.pl.umap(adata_mp, color=['batch'])

In [ ]:
sc.pl.umap(adata_mp, color=['state'])

In [ ]:
pd.crosstab(adata_mp.obs['state'], adata_mp.obs['batch'])

In [ ]:
celltypist.models.models_description()

In [ ]:
adata_mp_raw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_mp02.h5ad")
adata_mp_raw.obs_names_make_unique()

In [ ]:
predictions = celltypist.annotate(adata_mp_raw, model = 'Immune_All_Low.pkl', majority_voting = True)

In [ ]:
adata_mp_raw = predictions.to_adata()
adata_mp.obs['celltypist_Immune'] = adata_mp_raw.obs['majority_voting']
sc.pl.umap(adata_mp, color = 'celltypist_Immune')

In [ ]:
sc.pl.tsne(adata_mp, color = ['batch','celltypist_Immune'])

In [ ]:
sc.pl.umap(adata_mp, color=['VSIG4','CD5L'])

In [ ]:
sc.pl.tsne(adata_mp, color = ['SLC7A5'])

In [ ]:
pd.crosstab(adata_mp.obs['batch'], adata_mp.obs['celltypist_Immune'])

In [ ]:
adata_mp.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_mp03.h5ad")

In [ ]:
adata_mp=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/merged_adata_mp03.h5ad")

# Sc/Sn/Cosmx merged analysis v02

In [ ]:
# 각 데이터를 합치기 전에 normalization과 log변환을 진행 후 HVG 추출 이후 병합

In [ ]:
adata_scraw

In [ ]:
adata_snraw

In [ ]:
adata_spraw

In [ ]:
for ad in [adata_scraw, adata_snraw, adata_spraw]:
    sc.pp.normalize_total(ad, target_sum=1e4)
    sc.pp.log1p(ad)

In [ ]:
adata_merged = sc.concat(
    [adata_scraw, adata_snraw, adata_spraw],
    join='inner',       # ✅ 공통 유전자만
    label='batch',      # batch 구분 컬럼
    keys=['scRNA', 'snRNA', 'CosMx'],
    index_unique=None
)
print(adata_merged.shape)

In [ ]:
sc.pp.highly_variable_genes(
    adata_merged,
    n_top_genes=2000,
    flavor='seurat_v3',
    batch_key='batch'
)

In [ ]:
adata_merged.obs_names_make_unique

In [ ]:
adata_merged.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/sc_sn_sp/merged_adata_01.h5ad")

In [ ]:
sc.pp.scale(adata_merged, max_value=10)
sc.tl.pca(adata_merged, svd_solver='arpack')
print(adata_merged.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_merged, color='batch')

import matplotlib.pyplot as plt
var_ratio = adata_merged.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata_merged, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_merged, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_merged)
sc.pl.umap(adata_merged, color=['batch'])

In [ ]:
adata_merged.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/sc_sn_sp/merged_adata_02.h5ad")

In [ ]:
adata_merged=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/sc_sn_sp/merged_adata_02.h5ad")

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_merged.obsm['X_pca'],adata_merged.obs, 'batch')

# Harmony 결과를 저장
adata_merged.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_merged, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_merged)
sc.pl.umap(adata_merged, color=['batch'])

In [ ]:
adata_merged.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/sc_sn_sp/merged_adata_hm01.h5ad")

In [ ]:
adata_merged=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/sc_sn_sp/merged_adata_hm01.h5ad")

In [ ]:
adata_merged

In [ ]:
sc.pl.umap(
    adata_merged,
    color='batch',
    frameon=False,
    title='Analysis method',
    size=3,
    show=False,        # 이걸 False로 해야 plt.savefig()가 동작
    #return_fig=True    # <- 중요: 그림 객체 반환
)
# 직접 저장
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_IL25.pdf", bbox_inches='tight', dpi=300)
#plt.close()

In [ ]:
sc.pl.umap(
    adata_merged,
    color='PTPRC',
    frameon=False,
    #title='Lymphoid',
    size=3,
)

In [ ]:
sc.pl.umap(
    adata_merged,
    color='MKI67',
    frameon=False,
    #title='Lymphoid',
    size=3,
)

In [ ]:
sc.pl.umap(
    adata_merged,
    color=['VSIG4','CD5L'],
    frameon=False,
    #title='Lymphoid',
    size=3,
)

In [ ]:
sc.pl.umap(
    adata_merged,
    color=['CD3D','CD3E','CD3G'],
    frameon=False,
    #title='T cell',
    size=3,
)

In [ ]:
sc.pl.umap(
    adata_merged,
    color=['CD5L','FOLR2'],
    frameon=False,
    #title='T cell',
    size=3,
)

In [ ]:
adata_sc_only = adata_scraw.copy()
adata_sc_only.obs_names_make_unique

In [ ]:
sc.pp.normalize_total(adata_sc_only, target_sum=1e4)
sc.pp.log1p(adata_sc_only)

In [ ]:
# obs_names 고유화
adata_merged.obs_names_make_unique()
adata_scraw.obs_names_make_unique()

# 공통 셀만 추출
common_cells = adata_scraw.obs_names.intersection(adata_merged.obs_names)

# 슬라이싱
adata_sc_only = adata_scraw[common_cells, :].copy()
adata_sc_only.obsm['X_umap'] = adata_merged[common_cells, :].obsm['X_umap']

In [ ]:
adata_sn_only = adata_snraw.copy()
adata_sn_only.obs_names_make_unique

sc.pp.normalize_total(adata_sn_only, target_sum=1e4)
sc.pp.log1p(adata_sn_only)

In [ ]:
# obs_names 고유화
adata_merged.obs_names_make_unique()
adata_snraw.obs_names_make_unique()

# 공통 셀만 추출
common_cells = adata_snraw.obs_names.intersection(adata_merged.obs_names)

# 슬라이싱
adata_sn_only = adata_snraw[common_cells, :].copy()
adata_sn_only.obsm['X_umap'] = adata_merged[common_cells, :].obsm['X_umap']

In [ ]:
celltypist.models.models_description()

In [ ]:
sc.pl.umap(adata_sc_only, color = 'state')

In [ ]:
pred = celltypist.annotate(
    adata_sc_only,
    model="Healthy_Human_Liver.pkl",
    majority_voting=True
)

In [ ]:
adata_sc_only = pred.to_adata()
adata_sc_only.obs['celltypist_Liver'] = adata_sc_only.obs['majority_voting']
sc.pl.umap(adata_sc_only, color = 'celltypist_Liver')

In [ ]:
adata_merged

In [ ]:
sc.pl.umap(
    adata_merged,
    color=['state'],
    frameon=False,
    #title='T cell',
    size=3,
)

In [ ]:
adata_merged.obs['state_specific'] = adata_merged.obs['state']
adata_merged.obs['state'] = adata_merged.obs['state'].replace({
    'Normal': 'Healthy',
    'Fibrosis': 'MASLD',
    'Inflammation1': 'MASLD',
    'Inflammation2': 'MASLD',
    'NAFLD': 'MASLD',
    'Normal_DM': 'Healthy',
    'Steatosis': 'MASLD',
    'healthy': 'Healthy',
    'normal': 'Healthy'
})
sc.pl.umap(adata_merged, color=['state'])

In [ ]:
sc.pl.umap(
    adata_merged,
    color=['state'],
    frameon=False,
    size=3
)

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 차원 축소된 데이터 (예: PCA or UMAP 등)
X = adata_merged.obsm['X_umap']  # 또는 'X_umap' 등

inertias = []
k_range = range(2, 20)  

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)  # inertia = 클러스터 내부 거리 제곱합

# Plot
plt.figure(figsize=(6, 4))
plt.plot(k_range, inertias, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.grid(True)
plt.show()

In [ ]:
sc.tl.leiden(adata_merged, resolution=0.5, key_added="cluster")
sc.pl.umap(adata_merged, color=['cluster'])

In [ ]:
sc.pl.umap(
    adata_merged,
    color=['cluster'],
    frameon=False,
    size=3
)

In [ ]:
sc.pl.umap(
    adata_merged,
    color=['CD5L','FOLR2','C1QC','C1QB'],
    frameon=False,
    size=3
)

In [ ]:
adata_merged.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/sc_sn_sp/merged_adata_hm02.h5ad")

In [ ]:
adata_merged=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/sc_sn_sp/merged_adata_hm02.h5ad")

In [ ]:
genes_str = """
NDST3
CD5L
LILRB5
SDC3
FOLR2
C1QC
C1QB
CTD-2337J16.1
SIGLEC1
C1QA
VCAM1
MARCO
TIMD4
AXL
CXCL12
CETP
IGSF21
LYVE1
GFRA2
SLC40A1
GPNMB
APOE
LGMN
MSR1
GPR34
MRC1
VSIG4
PLTP
FPR3
CD163
EBI3
SEPP1
ADORA3
MS4A4A
MRC1L1
ETV5
SLC1A3
RND3
TNFRSF21
PLA2G7
ADAP2
RGL1
C2
FMNL2
FILIP1L
MERTK
DAB2
CR1
MS4A7
CMKLR1
"""

In [ ]:
# 줄 단위로 나눠서 리스트 만들기
gene_list = [g.strip() for g in genes_str.strip().split("\n") if g.strip()]

# 원하는 변수에 저장
Hepa_genes = gene_list

print(Hepa_genes)

In [ ]:
gene_sets = {
    "Hepa_genes": [ 'CYP2C8', 'CYP3A4', 'CYP2E1', 'CP', 'CYP3A43', 'CUX2', 'SORBS2', 'SDS', 'ZNF385D', 'ERRFI1', 'RNF152', 'SLC27A2', 'SLC16A12', 'C3', 'F5', 'GREM2', 'MAOB', 'CYP3A5', 'AASS', 'LIPC', 'NOS1AP', 'UPB1', 'GPLD1', 'ARG1', 'TENM1', 'FAM9B', 'COBLL1', 'ALDOB', 'RP11-432I5.1', 'FMO5', 'GPAM', 'CDHR3', 'MARC2', 'IGFBP1', 'GPR126', 'OSBPL6', 'NEK10', 'CYP4F3', 'PLIN5', 'SLC1A2', 'TMEM150C', 'CXADR', 'IGFBP2', 'CMYA5', 'TFR2', 'NECAB2', 'ADAMTS17', 'NRG1', 'FGA', 'RP11-431K24.1'],
    "Chola_genes": ['DCDC2', 'SCGN', 'CFTR', 'BICC1', 'ITGB8', 'KRT19', 'CTNND2', 'CLDN10', 'KRT7', 'SNAP25', 'DEFB1', 'MUC6', 'CASR', 'FAM150B', 'SEMA3E', 'PKHD1', 'FXYD2', 'CCDC64B', 'ELF3', 'HUNK', 'STK33', 'CASC15', 'KCNJ15', 'FGFR2', 'KRT8', 'CALN1', 'STXBP6', 'EPB41L1', 'LINC00671', 'PRDM16', 'SLC38A11', 'ANXA3', 'SEMA6D', 'C1orf186', 'ATP13A4', 'MACC1', 'SLC35F3', 'FLRT2', 'LINC00511', 'LGALS4', 'TACSTD2', 'KLF5', 'PMEPA1', 'ANXA4', 'SCD5', 'MAP2', 'BACE2', 'FAM171A1', 'CNTN4', 'TTN'],
    "cDC2_genes": ['FCER1A', 'CD1C', 'CD1E', 'CLEC10A', 'PPP1R14A', 'HLA-DQA1', 'HLA-DRA', 'CSF2RA', 'PKIB', 'PLD4', 'HLA-DQB1', 'HLA-DPB1', 'IL1R2', 'LGALS2', 'CD74', 'RNASE6', 'HLA-DRB5', 'HLA-DPA1', 'HLA-DRB1', 'CST3', 'CLIC2', 'SDPR', 'PHACTR1', 'FILIP1L', 'P2RY14', 'IFI30', 'LY86', 'IL1B', 'CD86', 'GAPT', 'C15orf48', 'HLA-DMB', 'LYZ', 'FCGR2B', 'CXorf21', 'ADAM28', 'IL18', 'SGK1', 'IL13RA1', 'GPR183', 'CPVL', 'HLA-DMA', 'KLF4', 'RAB32', 'NR4A3', 'CFP', 'IGSF6', 'PAK1', 'AXL', 'PLBD1'],
    "cDC_genes":['CCL19', 'CCL22', 'LAMP3', 'LAD1', 'CCR7', 'FSCN1', 'IDO1', 'CCL17', 'EBI3', 'CD1E', 'C15orf48', 'AC079767.4', 'DUSP4', 'CD200', 'CSF2RA', 'MARCKSL1', 'IL4I1', 'RAMP1', 'PTGIR', 'CLIC2', 'MGLL', 'SPIB', 'BIRC3', 'DAPP1', 'CD83', 'HLA-DOB', 'TXN', 'HLA-DQA1', 'NRP2', 'CD1C', 'HLA-DRA', 'C12orf45', 'MARCKS', 'FABP5', 'GSN', 'LGALS2', 'MIR155HG', 'SERPINF1', 'HILPDA', 'RASSF4', 'HLA-DQB1', 'CTSH', 'CLECL1', 'CD74', 'HLA-DPB1', 'BASP1', 'TNFAIP2', 'GPR183', 'FILIP1L', 'NR4A3'],
    "Mono_genes":['CD300E', 'FCN1', 'RP11-290F20.3', 'EREG', 'LILRA5', 'APOBEC3A', 'LILRB2', 'CSTA', 'VCAN', 'IFI30', 'FCGR1A', 'THBS1', 'IL1B', 'SLC7A7', 'LRRC25', 'AIF1', 'SLC11A1', 'HCK', 'SPI1', 'HBEGF', 'LYZ', 'PRAM1', 'MAFB', 'NCF2', 'LST1', 'RP11-1143G9.4', 'LGALS2', 'PILRA', 'CLEC7A', 'PLAUR', 'THBD', 'CYBB', 'CLEC4E', 'CFD', 'CD86', 'EMR2', 'S100A9', 'CLEC12A', 'KCTD12', 'PLBD1', 'SERPINA1', 'KLF4', 'C5AR1', 'CD68', 'BTK', 'TLR2', 'IFITM3', 'CSF1R', 'FCGR2A', 'FPR2'],
    "Neutrophil_genes":['FCGR3B', 'CMTM2', 'S100P', 'PTGS2', 'CXCR2', 'S100A8', 'G0S2', 'S100A12', 'CSF3R', 'CTB-61M7.2', 'MNDA', 'IL8', 'S100A9', 'FPR1', 'IL1R2', 'FPR2', 'MXD1', 'TREM1', 'VNN2', 'BASP1', 'LILRB3', 'GCA', 'NAMPT', 'CDA', 'PLBD1', 'LRRK2', 'SELL', 'CLEC4E', 'NCF2', 'C5AR1', 'RGS18', 'TRIB1', 'CD93', 'RBP7', 'SLC25A37', 'TMEM154', 'EGR1', 'AQP9', 'SLC11A1', 'VCAN', 'TLR2', 'CSTA', 'MEGF9', 'PLAUR', 'SOD2', 'RGS2', 'APOBEC3A', 'IRAK3', 'LYZ', 'RP11-1143G9.4'],
    "Basophil_genes":['TPSAB1', 'CMA1', 'CPA3', 'RP11-354E11.2', 'CTSG', 'MS4A2', 'GATA2', 'HPGDS', 'GSTM5', 'HDC', 'KIT', 'RGS13', 'KRT1', 'GCSAML', 'SLC18A2', 'IL1RL1', 'VWA5A', 'C1orf186', 'CNRIP1', 'LTC4S', 'HPGD', 'STXBP6', 'SLC45A3', 'BACE2', 'GSTM3', 'CD9', 'FCER1A', 'SDPR', 'CTD-3203P2.2', 'RAB27B', 'CLU', 'ACOT7', 'TIMP3', 'ALOX5', 'LMNA', 'SOX4', 'SLC26A2', 'BTK', 'CAPG', 'RP11-620J15.3', 'TSC22D1', 'RAB32', 'STX3', 'MAOB', 'STMN1', 'AREG', 'CPM', 'CD82', 'SERPINB1', 'LMO4'],
    "NK_genes":['XCL1', 'KLRC1', 'IL2RB', 'NCR1', 'TOX2', 'SPRY2', 'CD160', 'XCL2', 'SH2D1B', 'KLRF1', 'TXK', 'CCL3', 'SLFN13', 'TMIGD2', 'CLIC3', 'NCAM1', 'CD38', 'P2RY11', 'EOMES', 'MCTP2', 'CD7', 'CMC1', 'SPRY1', 'LAT2', 'FASLG', 'PTGDR', 'AREG', 'TIGIT', 'MATK', 'TOX', 'CD247', 'ABCB1', 'KLRD1', 'CCL4', 'CXCR6', 'S1PR5', 'RP11-222K16.2', 'IFITM1', 'COG2', 'APBA2', 'RP11-489E7.4', 'KLRB1', 'IL18', 'NKG7', 'IRF8', 'FCER1G', 'CXXC5', 'IFNG', 'SH2D1A', 'SYK'],
    "T_genes":['IL7R', 'CD40LG', 'CD3D', 'CD3G', 'CD8B', 'TRAT1', 'CD8A', 'SIT1', 'GPR171', 'CD3E', 'BCL11B', 'CD6', 'GZMK', 'LAG3', 'SPOCK2', 'ITM2A', 'IL32', 'CD2', 'AC092580.4', 'CXCR6', 'KLRG1', 'TC2N', 'CD27', 'LTB', 'ITK', 'CD69', 'RASGRP1', 'CCL5', 'KLRB1', 'GZMH', 'SH2D1A', 'RHOH', 'NCR3', 'SCML4', 'GZMA', 'RP11-138A9.2', 'MYBL1', 'LAT', 'RP11-94L15.2', 'CD96', 'PRR5', 'SH2D2A', 'DUSP2', 'GZMM', 'TUBA4A', 'ODF2L', 'CST7', 'PTPN7', 'LYAR', 'LBH'],
    "pDC_genes":['SCT', 'LRRC26', 'LILRA4', 'MAP1A', 'PTCRA', 'LAMP5', 'CLEC4C', 'MYBL2', 'SMIM5', 'KCNK17', 'PACSIN1', 'RP11-117D22.2', 'TCL1A', 'IL3RA', 'SPIB', 'PLD4', 'BCL11A', 'RASD1', 'FAM129C', 'TPM2', 'TSPAN13', 'PTPRS', 'SERPINF1', 'PPP1R14A', 'DERL3', 'SMPD3', 'IRF4', 'ITM2C', 'PFKFB2', 'PTGDS', 'FUT7', 'SOX4', 'IRF7', 'CXCR3', 'P2RY14', 'NOTCH4', 'PMEPA1', 'LINC00996', 'PPP1R14B', 'CLN8', 'VIPR2', 'AEBP1', 'C1orf186', 'MZB1', 'IRF8', 'TCF4', 'UGCG', 'GZMB', 'TNFRSF21', 'GAPT'],
    "B_genes":['VPREB3', 'FCRL1', 'RP11-693J15.5', 'LINC00926', 'FCRLA', 'FCRL2', 'CD19', 'CD79A', 'PAX5', 'RP5-887A10.1', 'AC079767.4', 'MS4A1', 'BANK1', 'CTA-250D10.23', 'CD22', 'TNFRSF13B', 'TNFRSF13C', 'KIAA0125', 'FCRL5', 'FCER2', 'TCL1A', 'FAM129C', 'POU2AF1', 'HLA-DOB', 'BLK', 'SPIB', 'CD79B', 'ADAM28', 'CCR7', 'BCL11A', 'BTLA', 'ARHGAP24', 'HVCN1', 'MARCH1', 'GNG7', 'CD72', 'IFT57', 'RIC3', 'SWAP70', 'P2RX5', 'MEF2C', 'BTK', 'GAPT', 'BLNK', 'DAPP1', 'RALGPS2', 'AFF3', 'CXXC5', 'SELL', 'FCGR2B'],
    "Plasma_genes":['RP11-731F5.2', 'IGJ', 'IGLL5', 'TNFRSF17', 'RP11-16E12.2', 'MZB1', 'DERL3', 'AL928768.3', 'POU2AF1', 'FCRL5', 'AC104024.1', 'SPAG4', 'JSRP1', 'KIAA0125', 'TNFRSF13B', 'ABCB9', 'BHLHE41', 'SSPN', 'CCR10', 'PNOC', 'AC006129.4', 'RP5-887A10.1', 'CD79A', 'SDC1', 'ITM2C', 'PRDX4', 'CAV1', 'AC093818.1', 'SEC11C', 'HLA-DOB', 'GNG7', 'SSR4', 'RAB30', 'TNFRSF13C', 'HBB', 'PPAPDC1B', 'HERPUD1', 'FKBP11', 'CD38', 'QPCT', 'CD27', 'XBP1', 'C16orf74', 'HSP90B1', 'PDK1', 'PIM2', 'MEI1', 'FAM46C', 'SEL1L3', 'LRRC16A'],
    "cDC1_genes":['CLEC9A', 'TACSTD2', 'IDO1', 'WDFY4', 'C1orf54', 'HLA-DOB', 'CLNK', 'CPVL', 'EGLN3', 'DNASE1L3', 'KIAA0226L', 'LGALS2', 'CST3', 'CCND1', 'S100B', 'HLA-DPB1', 'CD74', 'DUSP4', 'HLA-DRA', 'HLA-DPA1', 'HLA-DQB1', 'C15orf48', 'HLA-DQA1', 'HLA-DRB5', 'SNX3', 'RAB32', 'P2RY14', 'HLA-DRB1', 'RGCC', 'CSF2RA', 'IRF8', 'SERPINF1', 'BTLA', 'LMNA', 'ADAM28', 'TSPAN13', 'ASB2', 'CLIC2', 'AIM2', 'KCNK6', 'BASP1', 'MPEG1', 'C12orf45', 'LYZ', 'CXCL16', 'RGS10', 'VMO1', 'DAPP1', 'LY86', 'SEMA4A'],
    "MAC_genes":['NDST3', 'CD5L', 'LILRB5', 'SDC3', 'FOLR2', 'C1QC', 'C1QB', 'CTD-2337J16.1', 'SIGLEC1', 'C1QA', 'VCAM1', 'MARCO', 'TIMD4', 'AXL', 'CXCL12', 'CETP', 'IGSF21', 'LYVE1', 'GFRA2', 'SLC40A1', 'GPNMB', 'APOE', 'LGMN', 'MSR1', 'GPR34', 'MRC1', 'VSIG4', 'PLTP', 'FPR3', 'CD163', 'EBI3', 'SEPP1', 'ADORA3', 'MS4A4A', 'MRC1L1', 'ETV5', 'SLC1A3', 'RND3', 'TNFRSF21', 'PLA2G7', 'ADAP2', 'RGL1', 'C2', 'FMNL2', 'FILIP1L', 'MERTK', 'DAB2', 'CR1', 'MS4A7', 'CMKLR1']}


In [ ]:
gene_sets["Hepa_genes"] 

In [ ]:
sc.tl.score_genes(adata_merged, gene_list=gene_sets["Hepa_genes"] , score_name="Hepa_score")

In [ ]:
sc.pl.umap(adata_merged, color='Hepa_score', cmap='bwr', title='Hepa_score')

In [ ]:
sc.tl.score_genes(adata_merged, gene_list=gene_sets["Mono_genes"] , score_name="Mono_score")
sc.pl.umap(adata_merged, color='Mono_score', cmap='bwr', title='Mono_score')

In [ ]:
sc.tl.score_genes(adata_merged, gene_list=gene_sets["MAC_genes"] , score_name="MAC_score")
sc.pl.umap(adata_merged, color='MAC_score', cmap='bwr', title='MAC_score')

In [ ]:
sc.tl.score_genes(adata_merged, gene_list=gene_sets["NK_genes"] , score_name="NK_score")
sc.pl.umap(adata_merged, color='NK_score', cmap='bwr', title='NK_score')

In [ ]:
sc.tl.score_genes(adata_merged, gene_list=gene_sets["Chola_genes"] , score_name="Chola_score")
sc.pl.umap(adata_merged, color='Chola_score', cmap='bwr', title='Chola_score')

In [ ]:
sc.tl.score_genes(adata_merged, gene_list=gene_sets["T_genes"] , score_name="T_score")
sc.pl.umap(adata_merged, color='T_score', cmap='bwr', title='T_score')

In [ ]:
sc.tl.score_genes(adata_merged, gene_list=gene_sets["B_genes"] , score_name="B_score")
sc.pl.umap(adata_merged, color='B_score', cmap='bwr', title='B_score')

In [ ]:
sc.tl.score_genes(adata_merged, gene_list=gene_sets["pDC_genes"] , score_name="pDC_score")
sc.pl.umap(adata_merged, color='pDC_score', cmap='bwr', title='pDC_score')

In [ ]:
gene_sets = {
    "Hepa_genes": [ 'CYP2C8', 'CYP3A4', 'CYP2E1', 'CP', 'CYP3A43', 'CUX2', 'SORBS2', 'SDS', 'ZNF385D', 'ERRFI1', 'RNF152', 'SLC27A2', 'SLC16A12', 'C3', 'F5', 'GREM2', 'MAOB', 'CYP3A5', 'AASS', 'LIPC', 'NOS1AP', 'UPB1', 'GPLD1', 'ARG1', 'TENM1', 'FAM9B', 'COBLL1', 'ALDOB', 'RP11-432I5.1', 'FMO5', 'GPAM', 'CDHR3', 'MARC2', 'IGFBP1', 'GPR126', 'OSBPL6', 'NEK10', 'CYP4F3', 'PLIN5', 'SLC1A2', 'TMEM150C', 'CXADR', 'IGFBP2', 'CMYA5', 'TFR2', 'NECAB2', 'ADAMTS17', 'NRG1', 'FGA', 'RP11-431K24.1'],
    "Chola_genes": ['DCDC2', 'SCGN', 'CFTR', 'BICC1', 'ITGB8', 'KRT19', 'CTNND2', 'CLDN10', 'KRT7', 'SNAP25', 'DEFB1', 'MUC6', 'CASR', 'FAM150B', 'SEMA3E', 'PKHD1', 'FXYD2', 'CCDC64B', 'ELF3', 'HUNK', 'STK33', 'CASC15', 'KCNJ15', 'FGFR2', 'KRT8', 'CALN1', 'STXBP6', 'EPB41L1', 'LINC00671', 'PRDM16', 'SLC38A11', 'ANXA3', 'SEMA6D', 'C1orf186', 'ATP13A4', 'MACC1', 'SLC35F3', 'FLRT2', 'LINC00511', 'LGALS4', 'TACSTD2', 'KLF5', 'PMEPA1', 'ANXA4', 'SCD5', 'MAP2', 'BACE2', 'FAM171A1', 'CNTN4', 'TTN'],
    "cDC2_genes": ['FCER1A', 'CD1C', 'CD1E', 'CLEC10A', 'PPP1R14A', 'HLA-DQA1', 'HLA-DRA', 'CSF2RA', 'PKIB', 'PLD4', 'HLA-DQB1', 'HLA-DPB1', 'IL1R2', 'LGALS2', 'CD74', 'RNASE6', 'HLA-DRB5', 'HLA-DPA1', 'HLA-DRB1', 'CST3', 'CLIC2', 'SDPR', 'PHACTR1', 'FILIP1L', 'P2RY14', 'IFI30', 'LY86', 'IL1B', 'CD86', 'GAPT', 'C15orf48', 'HLA-DMB', 'LYZ', 'FCGR2B', 'CXorf21', 'ADAM28', 'IL18', 'SGK1', 'IL13RA1', 'GPR183', 'CPVL', 'HLA-DMA', 'KLF4', 'RAB32', 'NR4A3', 'CFP', 'IGSF6', 'PAK1', 'AXL', 'PLBD1'],
    "cDC_genes":['CCL19', 'CCL22', 'LAMP3', 'LAD1', 'CCR7', 'FSCN1', 'IDO1', 'CCL17', 'EBI3', 'CD1E', 'C15orf48', 'AC079767.4', 'DUSP4', 'CD200', 'CSF2RA', 'MARCKSL1', 'IL4I1', 'RAMP1', 'PTGIR', 'CLIC2', 'MGLL', 'SPIB', 'BIRC3', 'DAPP1', 'CD83', 'HLA-DOB', 'TXN', 'HLA-DQA1', 'NRP2', 'CD1C', 'HLA-DRA', 'C12orf45', 'MARCKS', 'FABP5', 'GSN', 'LGALS2', 'MIR155HG', 'SERPINF1', 'HILPDA', 'RASSF4', 'HLA-DQB1', 'CTSH', 'CLECL1', 'CD74', 'HLA-DPB1', 'BASP1', 'TNFAIP2', 'GPR183', 'FILIP1L', 'NR4A3'],
    "Mono_genes":['CD300E', 'FCN1', 'RP11-290F20.3', 'EREG', 'LILRA5', 'APOBEC3A', 'LILRB2', 'CSTA', 'VCAN', 'IFI30', 'FCGR1A', 'THBS1', 'IL1B', 'SLC7A7', 'LRRC25', 'AIF1', 'SLC11A1', 'HCK', 'SPI1', 'HBEGF', 'LYZ', 'PRAM1', 'MAFB', 'NCF2', 'LST1', 'RP11-1143G9.4', 'LGALS2', 'PILRA', 'CLEC7A', 'PLAUR', 'THBD', 'CYBB', 'CLEC4E', 'CFD', 'CD86', 'EMR2', 'S100A9', 'CLEC12A', 'KCTD12', 'PLBD1', 'SERPINA1', 'KLF4', 'C5AR1', 'CD68', 'BTK', 'TLR2', 'IFITM3', 'CSF1R', 'FCGR2A', 'FPR2'],
    "Neutrophil_genes":['FCGR3B', 'CMTM2', 'S100P', 'PTGS2', 'CXCR2', 'S100A8', 'G0S2', 'S100A12', 'CSF3R', 'CTB-61M7.2', 'MNDA', 'IL8', 'S100A9', 'FPR1', 'IL1R2', 'FPR2', 'MXD1', 'TREM1', 'VNN2', 'BASP1', 'LILRB3', 'GCA', 'NAMPT', 'CDA', 'PLBD1', 'LRRK2', 'SELL', 'CLEC4E', 'NCF2', 'C5AR1', 'RGS18', 'TRIB1', 'CD93', 'RBP7', 'SLC25A37', 'TMEM154', 'EGR1', 'AQP9', 'SLC11A1', 'VCAN', 'TLR2', 'CSTA', 'MEGF9', 'PLAUR', 'SOD2', 'RGS2', 'APOBEC3A', 'IRAK3', 'LYZ', 'RP11-1143G9.4'],
    "Basophil_genes":['TPSAB1', 'CMA1', 'CPA3', 'RP11-354E11.2', 'CTSG', 'MS4A2', 'GATA2', 'HPGDS', 'GSTM5', 'HDC', 'KIT', 'RGS13', 'KRT1', 'GCSAML', 'SLC18A2', 'IL1RL1', 'VWA5A', 'C1orf186', 'CNRIP1', 'LTC4S', 'HPGD', 'STXBP6', 'SLC45A3', 'BACE2', 'GSTM3', 'CD9', 'FCER1A', 'SDPR', 'CTD-3203P2.2', 'RAB27B', 'CLU', 'ACOT7', 'TIMP3', 'ALOX5', 'LMNA', 'SOX4', 'SLC26A2', 'BTK', 'CAPG', 'RP11-620J15.3', 'TSC22D1', 'RAB32', 'STX3', 'MAOB', 'STMN1', 'AREG', 'CPM', 'CD82', 'SERPINB1', 'LMO4'],
    "NK_genes":['XCL1', 'KLRC1', 'IL2RB', 'NCR1', 'TOX2', 'SPRY2', 'CD160', 'XCL2', 'SH2D1B', 'KLRF1', 'TXK', 'CCL3', 'SLFN13', 'TMIGD2', 'CLIC3', 'NCAM1', 'CD38', 'P2RY11', 'EOMES', 'MCTP2', 'CD7', 'CMC1', 'SPRY1', 'LAT2', 'FASLG', 'PTGDR', 'AREG', 'TIGIT', 'MATK', 'TOX', 'CD247', 'ABCB1', 'KLRD1', 'CCL4', 'CXCR6', 'S1PR5', 'RP11-222K16.2', 'IFITM1', 'COG2', 'APBA2', 'RP11-489E7.4', 'KLRB1', 'IL18', 'NKG7', 'IRF8', 'FCER1G', 'CXXC5', 'IFNG', 'SH2D1A', 'SYK'],
    "T_genes":['IL7R', 'CD40LG', 'CD3D', 'CD3G', 'CD8B', 'TRAT1', 'CD8A', 'SIT1', 'GPR171', 'CD3E', 'BCL11B', 'CD6', 'GZMK', 'LAG3', 'SPOCK2', 'ITM2A', 'IL32', 'CD2', 'AC092580.4', 'CXCR6', 'KLRG1', 'TC2N', 'CD27', 'LTB', 'ITK', 'CD69', 'RASGRP1', 'CCL5', 'KLRB1', 'GZMH', 'SH2D1A', 'RHOH', 'NCR3', 'SCML4', 'GZMA', 'RP11-138A9.2', 'MYBL1', 'LAT', 'RP11-94L15.2', 'CD96', 'PRR5', 'SH2D2A', 'DUSP2', 'GZMM', 'TUBA4A', 'ODF2L', 'CST7', 'PTPN7', 'LYAR', 'LBH'],
    "pDC_genes":['SCT', 'LRRC26', 'LILRA4', 'MAP1A', 'PTCRA', 'LAMP5', 'CLEC4C', 'MYBL2', 'SMIM5', 'KCNK17', 'PACSIN1', 'RP11-117D22.2', 'TCL1A', 'IL3RA', 'SPIB', 'PLD4', 'BCL11A', 'RASD1', 'FAM129C', 'TPM2', 'TSPAN13', 'PTPRS', 'SERPINF1', 'PPP1R14A', 'DERL3', 'SMPD3', 'IRF4', 'ITM2C', 'PFKFB2', 'PTGDS', 'FUT7', 'SOX4', 'IRF7', 'CXCR3', 'P2RY14', 'NOTCH4', 'PMEPA1', 'LINC00996', 'PPP1R14B', 'CLN8', 'VIPR2', 'AEBP1', 'C1orf186', 'MZB1', 'IRF8', 'TCF4', 'UGCG', 'GZMB', 'TNFRSF21', 'GAPT'],
    "B_genes":['VPREB3', 'FCRL1', 'RP11-693J15.5', 'LINC00926', 'FCRLA', 'FCRL2', 'CD19', 'CD79A', 'PAX5', 'RP5-887A10.1', 'AC079767.4', 'MS4A1', 'BANK1', 'CTA-250D10.23', 'CD22', 'TNFRSF13B', 'TNFRSF13C', 'KIAA0125', 'FCRL5', 'FCER2', 'TCL1A', 'FAM129C', 'POU2AF1', 'HLA-DOB', 'BLK', 'SPIB', 'CD79B', 'ADAM28', 'CCR7', 'BCL11A', 'BTLA', 'ARHGAP24', 'HVCN1', 'MARCH1', 'GNG7', 'CD72', 'IFT57', 'RIC3', 'SWAP70', 'P2RX5', 'MEF2C', 'BTK', 'GAPT', 'BLNK', 'DAPP1', 'RALGPS2', 'AFF3', 'CXXC5', 'SELL', 'FCGR2B'],
    "Plasma_genes":['RP11-731F5.2', 'IGJ', 'IGLL5', 'TNFRSF17', 'RP11-16E12.2', 'MZB1', 'DERL3', 'AL928768.3', 'POU2AF1', 'FCRL5', 'AC104024.1', 'SPAG4', 'JSRP1', 'KIAA0125', 'TNFRSF13B', 'ABCB9', 'BHLHE41', 'SSPN', 'CCR10', 'PNOC', 'AC006129.4', 'RP5-887A10.1', 'CD79A', 'SDC1', 'ITM2C', 'PRDX4', 'CAV1', 'AC093818.1', 'SEC11C', 'HLA-DOB', 'GNG7', 'SSR4', 'RAB30', 'TNFRSF13C', 'HBB', 'PPAPDC1B', 'HERPUD1', 'FKBP11', 'CD38', 'QPCT', 'CD27', 'XBP1', 'C16orf74', 'HSP90B1', 'PDK1', 'PIM2', 'MEI1', 'FAM46C', 'SEL1L3', 'LRRC16A'],
    "cDC1_genes":['CLEC9A', 'TACSTD2', 'IDO1', 'WDFY4', 'C1orf54', 'HLA-DOB', 'CLNK', 'CPVL', 'EGLN3', 'DNASE1L3', 'KIAA0226L', 'LGALS2', 'CST3', 'CCND1', 'S100B', 'HLA-DPB1', 'CD74', 'DUSP4', 'HLA-DRA', 'HLA-DPA1', 'HLA-DQB1', 'C15orf48', 'HLA-DQA1', 'HLA-DRB5', 'SNX3', 'RAB32', 'P2RY14', 'HLA-DRB1', 'RGCC', 'CSF2RA', 'IRF8', 'SERPINF1', 'BTLA', 'LMNA', 'ADAM28', 'TSPAN13', 'ASB2', 'CLIC2', 'AIM2', 'KCNK6', 'BASP1', 'MPEG1', 'C12orf45', 'LYZ', 'CXCL16', 'RGS10', 'VMO1', 'DAPP1', 'LY86', 'SEMA4A'],
    "MAC_genes":['NDST3', 'CD5L', 'LILRB5', 'SDC3', 'FOLR2', 'C1QC', 'C1QB', 'CTD-2337J16.1', 'SIGLEC1', 'C1QA', 'VCAM1', 'MARCO', 'TIMD4', 'AXL', 'CXCL12', 'CETP', 'IGSF21', 'LYVE1', 'GFRA2', 'SLC40A1', 'GPNMB', 'APOE', 'LGMN', 'MSR1', 'GPR34', 'MRC1', 'VSIG4', 'PLTP', 'FPR3', 'CD163', 'EBI3', 'SEPP1', 'ADORA3', 'MS4A4A', 'MRC1L1', 'ETV5', 'SLC1A3', 'RND3', 'TNFRSF21', 'PLA2G7', 'ADAP2', 'RGL1', 'C2', 'FMNL2', 'FILIP1L', 'MERTK', 'DAB2', 'CR1', 'MS4A7', 'CMKLR1']}


In [ ]:
sc.pl.umap(
    adata_merged,
    color=['cluster'],
    frameon=False,
    size=3,
    legend_loc = 'on data'
)

In [ ]:
adata_merged

In [ ]:
pd.crosstab(adata_merged.obs['cluster'],adata_merged.obs['batch'])

In [ ]:
pd.crosstab(adata_merged.obs['state'],adata_merged.obs['batch'])

# mono/MAC analysis

In [ ]:
adata_merged=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/sc_sn_sp/merged_adata_hm02.h5ad")
adata_merged

In [ ]:
sc.pl.umap(
    adata_merged,
    color=['cluster'],
    frameon=False,
    size=3,
    legend_loc = 'on data'
)

In [ ]:
adata_mono = adata_merged[adata_merged.obs['cluster'] == '6'].copy()

In [ ]:
adata_mono

In [ ]:
adata_mono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/sc_sn_sp/merged_adata_mono01.h5ad")

In [ ]:
adata_mono=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/sc_sn_sp/merged_adata_mono01.h5ad")
adata_mono

In [ ]:
adata_mono.obs_names_make_unique()

In [ ]:
sc.pp.filter_genes(adata_mono, min_counts=1)
sc.pp.highly_variable_genes(
    adata_mono,
    flavor='seurat', 
    n_top_genes=2000
)

In [ ]:
adata_mono

In [ ]:
adata_mono = adata_mono[:, adata_mono.var['highly_variable']].copy()

In [ ]:
pd.crosstab(adata_mono.obs['state'],adata_mono.obs['batch'])

In [ ]:
sc.pp.scale(adata_mono, max_value=10)
sc.tl.pca(adata_mono, svd_solver='arpack')
print(adata_mono.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_mono, color='batch')

import matplotlib.pyplot as plt
var_ratio = adata_mono.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata_mono, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_mono, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_mono)
sc.pl.umap(adata_mono, color=['batch'])

In [ ]:
import harmonypy as hm

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_mono.obsm['X_pca'],adata_mono.obs, 'batch')

# Harmony 결과를 저장
adata_mono.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_mono, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_mono)
sc.pl.umap(adata_mono, color=['batch'])

In [ ]:
adata_mono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/sc_sn_sp/merged_adata_mono02.h5ad")

In [ ]:
adata_mono=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/sc_sn_sp/merged_adata_mono02.h5ad")

In [ ]:
sc.pl.umap(adata_mono, color=['state'])

In [ ]:
sc.pl.umap(
    adata_mono,
    color=['batch'],
    frameon=False,
    size=5,
    legend_loc = 'on data'
)

In [ ]:
sc.pl.umap(
    adata_mono,
    color=['VSIG4','CD5L'],
    frameon=False,
    #title='Lymphoid',
    size=5,
)

In [ ]:
genes_str = """
S100A8
S100A12
S100A9
VCAN
CTB-61M7.2
THBS1
EREG
MGST1
RP11-1143G9.4
ASGR1
BST1
AQP9
FCN1
CDA
LYZ
SLC2A3
MEGF9
APOBEC3A
CD300E
CD36
RAB27A
CLEC4E
TMEM176A
SELL
LGALS2
CXCL2
ITGAM
CSF3R
G0S2
EGR1
SOCS3
CCDC69
CAPG
TREM1
SLC11A1
AREG
"""
# 줄 단위로 나눠서 리스트 만들기
gene_list = [g.strip() for g in genes_str.strip().split("\n") if g.strip()]

print(gene_list)

In [ ]:
gene_sets = {
    "cDC1_genes": ['XCR1', 'CLNK', 'TACSTD2', 'CLEC9A', 'EGLN3', 'IDO1', 'DNASE1L3', 'CADM1', 'S100B', 'KIAA0226L', 'WDFY4', 'CCND1', 'DUSP4', 'P2RY14', 'C1orf54', 'NRARP', 'CPNE3', 'SLAMF7', 'GPR157', 'B3GNT7', 'FLT3', 'NET1', 'RGCC', 'SERPINF1', 'BATF3', 'NDRG2', 'SNX3', 'ADAM28', 'AIM2', 'IL18R1', 'GPR171', 'PTPRCAP', 'IRF8', 'CPVL', 'C12orf45', 'TPD52', 'SLC38A1', 'LMNA', 'CD59', 'C15orf48', 'NFATC2', 'NAV1', 'MAP2K6', 'PTMS', 'ARHGAP5', 'HLA-DPB1', 'CST7', 'LGALS2', 'HLA-DQB1', 'PPA1', 'TCTN3', 'APOL1', 'CAMK2D', 'HLA-DPA1', 'CD74', 'HLA-DQA1', 'CST3', 'BCL2L11', 'DHRS3', 'ID2', 'RAB11FIP1', 'QPRT', 'OSBPL9', 'PTPN22', 'PTPN7', 'FAM129A', 'DST', 'FUCA1', 'ANPEP', 'MARCKSL1', 'HLA-DRB1', 'CLIC2', 'STK17A', 'RGS1', 'HLA-DRA', 'BASP1', 'ACTN1', 'PDLIM7', 'CD40', 'AC093673.5', 'NCOA7', 'PDE4DIP', 'TUBA1C', 'STMN1', 'CSF2RA', 'CD38', 'MIR4435-1HG', 'DUSP2', 'LGMN', 'PKIB', 'VMO1', 'SPATS2L'],
    "cDC2_genes": ['FCER1A', 'PPP1R14A', 'CD1C', 'AKR1C3', 'ENHO', 'CD1E', 'SLC38A1', 'P2RY14', 'AFF3', 'FCGR2B', 'CLEC10A', 'NDRG2', 'CST7', 'PKIB', 'CACNA2D3', 'ADAM28', 'SDPR', 'PLD4', 'NET1', 'FLT3', 'GAS6', 'ALOX5AP', 'AGPAT9', 'HLA-DQA1', 'RGS1', 'FILIP1L', 'C12orf45', 'HLA-DQB1', 'IL1R2', 'ARHGAP5', 'HLA-DPB1', 'GPR183', 'AXL', 'ADAM8', 'USP53', 'SERPINF1', 'RGCC', 'HLA-DPA1', 'CD72', 'GSN', 'CLIC2', 'ACAP1', 'ATP1B1', 'HLA-DRA', 'PMAIP1', 'CD74', 'CSF2RA', 'AREG', 'PPA1', 'FPR3', 'HLA-DRB1', 'ALCAM', 'SPATS2L', 'ARL4C', 'PTMS', 'STMN1', 'LMNA', 'VSIG4'],
    "pDC_genes":['SCT', 'LRRC26', 'LILRA4', 'MAP1A', 'PTCRA', 'LAMP5', 'CLEC4C', 'MYBL2', 'SMIM5', 'KCNK17', 'PACSIN1', 'RP11-117D22.2', 'TCL1A', 'IL3RA', 'SPIB', 'PLD4', 'BCL11A', 'RASD1', 'FAM129C', 'TPM2', 'TSPAN13', 'PTPRS', 'SERPINF1', 'PPP1R14A', 'DERL3', 'SMPD3', 'IRF4', 'ITM2C', 'PFKFB2', 'PTGDS', 'FUT7', 'SOX4', 'IRF7', 'CXCR3', 'P2RY14', 'NOTCH4', 'PMEPA1', 'LINC00996', 'PPP1R14B', 'CLN8', 'VIPR2', 'AEBP1', 'C1orf186', 'MZB1', 'IRF8', 'TCF4', 'UGCG', 'GZMB', 'TNFRSF21', 'GAPT'],
    "MigcDC_genes": ['LAD1', 'CCR7', 'LAMP3', 'CCL19', 'CCL22', 'IL7R', 'SLC7A11', 'FSCN1', 'RAMP1', 'IDO1', 'CCL17', 'CD1B', 'MIR155HG', 'TNFRSF4', 'CXCL9', 'GPR157', 'DUSP4', 'IL4I1', 'EBI3', 'TRAF1', 'MARCKSL1', 'SPIB', 'ARHGAP10', 'HLA-DQB2', 'CD1E', 'MGLL', 'IL32', 'DUSP5', 'NRP2', 'CST7', 'PSD3', 'BIRC3', 'IRF4', 'C15orf48', 'HILPDA', 'GADD45A', 'SLC38A1', 'TRAF5', 'C12orf45', 'TXN', 'RAB9A', 'TBC1D4', 'TUBA1C', 'RGS1', 'SERPINF1', 'NMRK1', 'PPA1', 'CD40', 'GSN', 'POGLUT1', 'CLIC2', 'NAV1', 'PVRL2', 'PTGIR', 'RP11-138A9.2', 'FABP5', 'ALCAM', 'FAM129A', 'NET1', 'CSF2RA', 'ANTXR2', 'UGCG', 'GPR137B', 'CD1C', 'ADAM8', 'RELB', 'PALLD', 'SLAMF7', 'ST3GAL6', 'RP11-356I2.4', 'GPR183', 'HLA-DQA1', 'NFKB1', 'FLT3', 'RP11-138A9.1', 'CERS6', 'DNASE1L3', 'P2RY8', 'CD83', 'MAFF', 'SESN1', 'CYB5A', 'PTMS', 'SDC2', 'CCL5', 'ATP1B1', 'ETV3', 'ATF5', 'HLA-DQB1', 'FILIP1L', 'PMAIP1', 'PIM3', 'ID2', 'MARCKS', 'EPSTI1', 'BHLHE40', 'SOCS1', 'BASP1', 'HLA-DPB1', 'CD74', 'APOC1', 'HLA-DRA', 'CD72', 'GBP1', 'NR4A3', 'HLA-DPA1', 'PLEK', 'LMNA', 'CLEC10A'],
    "Monocyte_genes":['S100A8', 'S100A12', 'S100A9', 'VCAN', 'CTB-61M7.2', 'THBS1', 'EREG', 'MGST1', 'RP11-1143G9.4', 'ASGR1', 'BST1', 'AQP9', 'FCN1', 'CDA', 'LYZ', 'SLC2A3', 'MEGF9', 'APOBEC3A', 'CD300E', 'CD36', 'RAB27A', 'CLEC4E', 'TMEM176A', 'SELL', 'LGALS2', 'CXCL2', 'ITGAM', 'CSF3R', 'G0S2', 'EGR1', 'SOCS3', 'CCDC69', 'CAPG', 'TREM1', 'SLC11A1', 'AREG'],
    "Patmono_genes":['LYPD2', 'CDKN1C', 'CKB', 'PTP4A3', 'CD79B', 'HES4', 'PPM1N', 'SYTL1', 'FCGR3A', 'LILRA3', 'RHOC', 'RP11-290F20.3', 'TESC', 'SPN', 'TCF7L2', 'TPPP3', 'SLC2A6', 'SLC44A2', 'RRAS', 'SNX18', 'OAS1', 'FAM110A', 'VMO1', 'MTSS1', 'LILRB2', 'PHF19', 'TBC1D10C', 'FPR2', 'LILRA5', 'IFITM2', 'SIGLEC10', 'MYO1G', 'PAG1', 'SIDT2', 'LTB', 'LST1', 'S1PR4', 'ITGAL', 'STXBP2', 'CNIH4', 'GCH1', 'IFITM3', 'KLF2'],
    "LAM_genes":['FN1', 'CST6', 'SPP1', 'MT1G', 'TREM2', 'GPNMB', 'SLC16A10', 'CD9', 'EMP1', 'APOC1', 'LMNA', 'C15orf48', 'CLEC5A', 'IL7R', 'FABP5', 'RNASE1', 'FBP1', 'SDS', 'RAB13', 'OLR1', 'APOE', 'RGCC', 'GPX3', 'TSC22D1', 'SLC39A8', 'CAPG', 'ACP5', 'MMP19', 'ALCAM', 'GSN', 'FPR3', 'PDE4DIP', 'MT1X', 'LHFPL2', 'SDC2', 'MITF', 'PLA2G7', 'LGALS3', 'PHLDA1', 'ANKRD28', 'AVPI1', 'PTMS', 'MARCO', 'B3GNT5', 'PLTP', 'GPR183', 'DAB2', 'CD59', 'RGS1', 'LGMN'],
    "MATLAM_genes":['TREM2', 'GPNMB', 'CD9', 'RNASE1', 'OTOA', 'SDS', 'A2M', 'APOC1', 'LGMN', 'FCGBP', 'FABP5', 'SLCO2B1', 'APOE', 'C1QC', 'PLA2G7', 'FPR3', 'ID3', 'DAB2', 'AXL', 'C1QB', 'PLTP', 'RGS1', 'LTC4S', 'CD72', 'GATM', 'MGLL', 'LMNA', 'ABCC5', 'GAS6', 'ANKH', 'STMN1', 'C1QA', 'ALCAM', 'MSR1', 'PKIB', 'PHLDA1', 'ATP1B1', 'GPR34', 'PTMS', 'RGCC', 'MGAT4A', 'FCGR2B', 'LIPA', 'ACP5', 'CTSL', 'MITF', 'MAF', 'EBI3', 'KCNMA1', 'MERTK', 'VSIG4', 'C15orf48', 'FILIP1L', 'SEPP1', 'SLC40A1', 'CD81', 'GPR183', 'SDC2', 'GSN', 'FOLR2', 'CD59', 'CD84', 'CPM', 'MS4A4A', 'ABCA1', 'HLA-DQA1', 'CLEC10A', 'CLIC2', 'ARL4C', 'RGL1', 'EPB41L2', 'MMP19', 'AVPI1', 'ALB', 'C1orf54', 'PDK4', 'CD1C', 'VMO1'],
    "MAC1_genes":['IGSF21', 'FCGBP', 'USP53', 'ID3', 'C1QC', 'A2M', 'SDS', 'PLTP', 'MRC1L1', 'SLCO2B1', 'LTC4S', 'CD9', 'FCGR2B', 'RGS1', 'VSIG4', 'FPR3', 'C1QB', 'GPR34', 'AXL', 'FILIP1L', 'PKIB', 'C1QA', 'CD72', 'CLEC9A', 'NDRG2', 'CLEC10A', 'GATM', 'ALB', 'GPR183', 'MSR1', 'HLA-DPA1', 'PAPOLG', 'DAB2', 'PLD4', 'LPAR6', 'PHLDA1', 'EBI3', 'F13A1', 'FCER1A', 'HLA-DQA1', 'CPM', 'HLA-DPB1', 'ALOX5AP', 'SERPINF1', 'HLA-DRB1', 'RCAN1', 'STMN1', 'HLA-DRA', 'CCL3L1', 'ATP1B1', 'HLA-DQB1', 'PRDM1', 'MAF', 'ALCAM', 'C1orf54', 'APOC1', 'PTMS', 'LGALS3BP', 'CLIC2', 'RGCC', 'SLC40A1', 'VMO1', 'SERPING1', 'EPB41L2', 'CCL3', 'FOLR2', 'CD1C', 'PDK4', 'SEPP1', 'APOE'],
    "moKC_genes":['RNASE1', 'IGSF21', 'FCGBP', 'LILRB5', 'SLCO2B1', 'C1QC', 'SEPP1', 'PLTP', 'MRC1L1', 'NRP1', 'A2M', 'C1QB', 'TMEM37', 'LYVE1', 'FRMD4A', 'GFRA2', 'APOE', 'FOLR2', 'GPR34', 'OLFML2B', 'C1QA', 'LGMN', 'MAF', 'ID3', 'CXCL12', 'AXL', 'MERTK', 'SLC40A1', 'DAB2', 'MRC1', 'DHRS3', 'PDK4', 'GATM', 'LTC4S', 'MS4A4A', 'GPNMB', 'APOC1', 'USP53', 'EBI3', 'FPR3', 'FILIP1L', 'ALB', 'STAB1', 'SDC3', 'SIGLEC1', 'KCNMA1', 'VSIG4', 'ANKH', 'PKIB', 'MSR1', 'VCAM1', 'FCHO2', 'CCL3', 'SERPING1', 'CLEC9A', 'LGALS3BP', 'CCL4', 'CCL3L1', 'CD72', 'STMN1', 'CPM', 'CD163', 'RGL1', 'ABCA1', 'CD5L', 'EPS8', 'C2', 'CLIC2', 'RGS1', 'SPRED1', 'ATP1B1', 'TSPAN4', 'RCAN1', 'LIPA', 'SMPDL3A', 'WWP1', 'FRMD4B', 'EPB41L2', 'CD81', 'TCF4', 'AP2A2', 'SPATS2L', 'CD84', 'DST', 'FMNL2', 'MS4A7', 'CD59', 'CTSL', 'PLD3', 'HLA-DPA1', 'PTMS', 'ARL4C', 'VMO1', 'C1orf54', 'EGR2', 'CLEC10A', 'FCGR3A'],
    "KC_genes":['TIMD4', 'SLC16A9', 'NDST3', 'KCNAB1', 'ITGAD', 'CD5L', 'VCAM1', 'SELENBP1', 'BCAM', 'CDH5', 'CETP', 'RND3', 'FEZ1', 'FABP3', 'SDC3', 'CXCL12', 'ITLN1', 'CTD-2337J16.1', 'LYVE1', 'SEPP1', 'TMEM37', 'MARCO', 'SCD', 'LILRB5', 'PPAP2B', 'ITGA9', 'IGF1', 'APOE', 'SLC7A8', 'EPAS1', 'SLC40A1', 'FRMD4A', 'SUCNR1', 'ACVRL1', 'IGFBP4', 'ME1', 'LGMN', 'SIGLEC1', 'SIGLEC11', 'GPR126', 'FOLR2', 'LAG3', 'ITGB5', 'NRP1', 'C1QB', 'AXL', 'MRC1', 'C1QA', 'IFI27', 'C1QC', 'C2', 'SLC46A1', 'SLCO2B1', 'ETV5', 'SCN1B', 'NR1H3', 'DHRS3', 'PDK4', 'CD209', 'CCL4L1', 'GPNMB', 'ARHGEF12', 'CD59', 'RGL1', 'ANKRD36C', 'CCL4', 'MAF', 'EPB41L2', 'GATM', 'CCND1', 'CTSF', 'IFT74', 'LIPA', 'A2M', 'MYO9A', 'ATP1B1', 'CCL4L2', 'IFIT1', 'SERPING1', 'SPATS2L', 'MERTK', 'FUCA1', 'ADORA3', 'SMPDL3A', 'CCL3', 'ABCA1', 'MRC1L1', 'ANKH', 'CMKLR1', 'DST', 'PLD3', 'PLTP', 'FSCN1', 'APOC1', 'MMD', 'QPRT', 'VSIG4', 'RHOBTB3', 'EXT1', 'DAB2', 'ALB', 'RCAN1', 'FAM213A', 'EBI3', 'GFRA2', 'CD82', 'CCL3L3', 'PLA2G15', 'WWP1', 'CD81', 'ADM', 'ABCG1', 'FXYD6', 'STOM', 'KCNMA1', 'IFIT2', 'CD38', 'PPP1R12B', 'APOC3', 'CTSL', 'LGALS3BP', 'CR1', 'MSR1', 'TNFRSF21', 'CD163', 'GCNT1', 'EPHX1', 'SASH1', 'FILIP1L', 'SNX24', 'CLIC2', 'IFIT3', 'C1orf54', 'FPR3', 'FRMD4B', 'LTC4S', 'MMP19', 'CREG1', 'HMOX1', 'ZNF189', 'SDPR', 'NCEH1', 'FMNL2', 'CCL3L1', 'ITGAV', 'TSPAN4', 'ALDH1A1', 'NFIA', 'CTSB', 'PDE4DIP', 'ARL4C', 'ARRDC3', 'STMN1', 'RBP7', 'MS4A4A', 'EGR2', 'CD84', 'TCF4', 'VMO1', 'DSC2', 'GPR34', 'FCHO2', 'SLC9A9', 'PLA2G7', 'ADAP2', 'SCARB2', 'IL10', 'FCGRT', 'PHACTR2', 'AP2A2', 'ELL2', 'ACP2', 'MS4A7', 'P4HA1', 'PDGFC', 'GPR137B', 'SGPP1', 'SPRED1', 'PKIB', 'IFI44L', 'PRNP', 'CD72', 'SIAH2', 'MS4A6A', 'EPS8', 'UGCG', 'IDH1', 'BLVRB', 'ICAM1', 'RGS1', 'CTSD', 'CD63', 'FCGR3A', 'MT1F', 'GIMAP5', 'KCNJ2']
    }


In [ ]:
sc.tl.score_genes(adata_mono, gene_list=gene_sets["pDC_genes"] , score_name="pDC_score")
sc.pl.umap(adata_mono, color='pDC_score', cmap='bwr', title='pDC_score')

In [ ]:
sc.tl.score_genes(adata_mono, gene_list=gene_sets["cDC1_genes"] , score_name="cDC1_score")
sc.pl.umap(adata_mono, color='cDC1_score', cmap='bwr', title='cDC1_score')

In [ ]:
sc.tl.score_genes(adata_mono, gene_list=gene_sets["cDC2_genes"] , score_name="cDC2_score")
sc.pl.umap(adata_mono, color='cDC2_score', cmap='bwr', title='cDC2_score')

In [ ]:
sc.tl.score_genes(adata_mono, gene_list=gene_sets["MigcDC_genes"] , score_name="MigcDC_score")
sc.pl.umap(adata_mono, color='MigcDC_score', cmap='bwr', title='MigcDC_score')

In [ ]:
sc.tl.score_genes(adata_mono, gene_list=gene_sets["Monocyte_genes"] , score_name="Monocyte_score")
sc.pl.umap(adata_mono, color='Monocyte_score', cmap='bwr', title='Monocyte_score')

In [ ]:
sc.tl.score_genes(adata_mono, gene_list=gene_sets["Patmono_genes"] , score_name="Patmono_score")
sc.pl.umap(adata_mono, color='Patmono_score', cmap='bwr', title='Patmono_score')

In [ ]:
sc.tl.score_genes(adata_mono, gene_list=gene_sets["LAM_genes"] , score_name="LAM_score")
sc.pl.umap(adata_mono, color='LAM_score', cmap='bwr', title='LAM_score')

In [ ]:
sc.tl.score_genes(adata_mono, gene_list=gene_sets["MATLAM_genes"] , score_name="MATLAM_score")
sc.pl.umap(adata_mono, color='MATLAM_score', cmap='bwr', title='MATLAM_score')

In [ ]:
sc.tl.score_genes(adata_mono, gene_list=gene_sets["MAC1_genes"] , score_name="MAC1_score")
sc.pl.umap(adata_mono, color='MAC1_score', cmap='bwr', title='MAC1_score')

In [ ]:
sc.tl.score_genes(adata_mono, gene_list=gene_sets["moKC_genes"] , score_name="moKC_score")
sc.pl.umap(adata_mono, color='moKC_score', cmap='bwr', title='moKC_score')

In [ ]:
sc.tl.score_genes(adata_mono, gene_list=gene_sets["KC_genes"] , score_name="KC_score")
sc.pl.umap(adata_mono, color='KC_score', cmap='bwr', title='KC_score')

In [ ]:
pd.crosstab(adata_mono.obs['cluster'],adata_mono.obs['batch'])

In [ ]:
pd.crosstab(adata_mono.obs['cluster'],adata_mono.obs['batch'])

In [ ]:
pd.crosstab(adata_mono.obs['state'],adata_mono.obs['batch'])

In [ ]:
adata_mono.obsm

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 차원 축소된 데이터 (예: PCA or UMAP 등)
X = adata_mono.obsm['X_pca_harmony']  # 또는 'X_umap' 등

inertias = []
k_range = range(2, 20)  

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)  # inertia = 클러스터 내부 거리 제곱합

# Plot
plt.figure(figsize=(6, 4))
plt.plot(k_range, inertias, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.grid(True)
plt.show()

In [ ]:
sc.tl.leiden(adata_mono, resolution=0.15, key_added="cluster")
sc.pl.umap(adata_mono, color=['cluster'])

In [ ]:
from sklearn.metrics import silhouette_score

labels = adata_mono.obs['cluster']
score = silhouette_score(adata_mono.obsm['X_pca_harmony'], labels)
print(f"Silhouette score: {score:.3f}")

In [ ]:
from sklearn.metrics import silhouette_score

labels = adata_mono.obs['cluster']
score = silhouette_score(adata_mono.obsm['X_pca_harmony'], labels)
print(f"Silhouette score: {score:.3f}")

In [ ]:
sc.pl.umap(
    adata_mono,
    color=['cluster'],
    frameon=False,
    size=8,
    legend_loc = 'on data'
)

In [ ]:
adata_mono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/merged/sc_sn_sp/merged_adata_mono02.h5ad")